# Phase 3.3 - Advanced LLM-Augmented Hybrid Decision Framework

Notebook nay trien khai Level 5 v2: LightGBM score + local MiniLM embedding + dynamic hybrid decision policy.

Muc tieu:
- So sanh truc tiep voi tuned Level 4 tu Phase 3.1b.
- Dung embedding MiniLM that neu chay Colab/GPU.
- Fit prototype/similarity tren train only.
- Tune candidate/threshold tren validation only.
- Bao cao ro Level 5 v2 thang Level 4 o 0/3, 1/3, 2/3, hay 3/3 cost settings.

## 0. Run Contract

Mac dinh `RUN_MODE = "sample_100k"` de khop cac output Phase 2, Phase 3.1b va Phase 3.2 hien co.

Final LLM claim chi hop le khi metadata ghi `embedding_backend_used = "minilm"`. Neu notebook fallback sang TF-IDF/SVD, ket qua chi dung de smoke test code flow.

In [1]:
from pathlib import Path
import gc
import json
import os
import subprocess
import sys
import warnings

import numpy as np
import pandas as pd

import matplotlib
if "COLAB_RELEASE_TAG" not in os.environ:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.special import expit, logit
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import average_precision_score, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.preprocessing import normalize

try:
    from IPython.display import display
except Exception:
    display = print

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 220)

SEED = 42
np.random.seed(SEED)

RUN_MODE = "sample_100k"  # "smoke", "sample_100k", "sample_200k", "sample_300k", or "full"
RUN_MODE_SAMPLE_ROWS = {"smoke": 10_000, "sample_100k": 100_000, "sample_200k": 200_000, "sample_300k": 300_000, "full": None}
if RUN_MODE not in RUN_MODE_SAMPLE_ROWS:
    raise ValueError(f"Unsupported RUN_MODE={RUN_MODE!r}")

RUN_OUTPUT_TAG = RUN_MODE
SAMPLE_ROWS = RUN_MODE_SAMPLE_ROWS[RUN_MODE]
SAMPLE_ROWS_LABEL = SAMPLE_ROWS if SAMPLE_ROWS is not None else "full"

SELECTED_RISK_MODEL = "lightgbm_balanced"
MODEL_FALLBACK_ORDER = ["lightgbm_balanced", "xgboost_magic_style", "lightgbm_magic_style", "xgboost_scale_pos_weight", "random_forest_balanced", "logistic_regression_balanced"]
LEVEL4_COMPARATOR_SELECTOR = "level4_tuned_guarded_selector"

COST_CONFIGS = {"Cost-A": {"alpha": 0.05, "beta": 1.0}, "Cost-B": {"alpha": 0.10, "beta": 2.0}, "Cost-C": {"alpha": 0.20, "beta": 5.0}}
GAMMA_GRID = [-0.20, -0.10, -0.05, 0.00, 0.05, 0.10, 0.20]
W_GRID = [-2.0, -1.0, -0.5, 0.0, 0.5, 1.0, 2.0]
THRESHOLD_GRID = np.unique(np.concatenate([np.linspace(0.0, 1.0, 201), np.array([0.001, 0.005, 0.01, 0.02, 0.03, 0.05, 0.10, 0.20, 0.30, 0.40, 0.60, 0.70, 0.80, 0.90, 0.95, 0.99])]))

EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
AUTO_INSTALL_SENTENCE_TRANSFORMERS = "COLAB_RELEASE_TAG" in os.environ
EMBEDDING_BATCH_SIZE = 32 if RUN_MODE.startswith("sample_") else (64 if RUN_MODE == "smoke" else 128)
TEXT_SERIALIZE_CHUNK = 50_000
TFIDF_MAX_FEATURES = 4096
TFIDF_SVD_COMPONENTS = 64

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ
DEFAULT_COLAB_PROJECT_ROOT = "/content/drive/MyDrive/BaoMatCuoiKy/LLM-Assisted_Cost-Sensitive" if IN_COLAB else ""
PROJECT_ROOT_OVERRIDE = os.environ.get("PROJECT_ROOT_OVERRIDE", DEFAULT_COLAB_PROJECT_ROOT).strip()

if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as exc:
        print(f"Google Drive mount skipped/failed: {exc}")

def looks_like_project_root(path: Path) -> bool:
    return (path / "notebooks").exists() and (path / "results").exists() and (path / "data").exists()

def find_project_root() -> Path:
    candidates = []
    if PROJECT_ROOT_OVERRIDE:
        candidates.append(Path(PROJECT_ROOT_OVERRIDE))
    cwd = Path.cwd()
    candidates.extend([cwd, cwd.parent, cwd / "LLM-Assisted_Cost-Sensitive", Path("/content/LLM-Assisted_Cost-Sensitive"), Path("/content/drive/MyDrive/BaoMatCuoiKy/LLM-Assisted_Cost-Sensitive"), Path("/content/drive/MyDrive/LLM-Assisted_Cost-Sensitive")])
    seen = set()
    for candidate in candidates:
        candidate = candidate.expanduser().resolve()
        if candidate in seen:
            continue
        seen.add(candidate)
        if candidate.exists() and looks_like_project_root(candidate):
            return candidate
    checked = "\n".join(f"- {p}" for p in candidates)
    raise FileNotFoundError("Cannot find project root. Set PROJECT_ROOT_OVERRIDE.\nChecked:\n" + checked)

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RAW_DIR = PROJECT_ROOT / "data" / "raw"
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"
EMBEDDINGS_DIR = PROJECT_ROOT / "artifacts" / "embeddings"
for directory in [RESULTS_DIR, FIGURES_DIR, EMBEDDINGS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

PATHS = {
    "train_transaction": RAW_DIR / "train_transaction.csv",
    "train_identity": RAW_DIR / "train_identity.csv",
    "baseline_scores": RESULTS_DIR / f"baseline_scores_{RUN_OUTPUT_TAG}.csv",
    "phase31b_candidate_metrics": RESULTS_DIR / f"phase31b_candidate_metrics_{RUN_OUTPUT_TAG}.csv",
    "phase31b_selected_policies": RESULTS_DIR / f"phase31b_selected_policies_{RUN_OUTPUT_TAG}.csv",
    "phase31b_candidate_thresholds": RESULTS_DIR / f"phase31b_candidate_thresholds_{RUN_OUTPUT_TAG}.csv",
    "five_level_tuned": RESULTS_DIR / f"five_level_comparison_tuned_{RUN_OUTPUT_TAG}.csv",
}
print(f"Project root: {PROJECT_ROOT}")
print(f"RUN_MODE={RUN_MODE} | SCORE_PATH={PATHS['baseline_scores']}")
print("Phase 3.3 scope: Advanced Level 5 v2 LLM-augmented hybrid decision policy.")


Mounted at /content/drive
Project root: /content/drive/MyDrive/BaoMatCuoiKy/LLM-Assisted_Cost-Sensitive
RUN_MODE=sample_100k | SCORE_PATH=/content/drive/MyDrive/BaoMatCuoiKy/LLM-Assisted_Cost-Sensitive/results/baseline_scores_sample_100k.csv
Phase 3.3 scope: Advanced Level 5 v2 LLM-augmented hybrid decision policy.


## 1. Load Phase 2, Phase 3.1b, and Optional Phase 3.2 Artifacts


In [2]:
REQUIRED_SCORE_COLUMNS = {"run_mode", "sample_rows", "split", "TransactionID", "isFraud", "TransactionAmt", "model", "score"}
REQUIRED_INPUTS = ["baseline_scores", "phase31b_candidate_metrics", "phase31b_selected_policies", "phase31b_candidate_thresholds", "five_level_tuned"]
missing_inputs = [name for name in REQUIRED_INPUTS if not PATHS[name].exists()]
if missing_inputs:
    available = sorted(str(p.relative_to(PROJECT_ROOT)) for p in RESULTS_DIR.glob("*.csv"))
    raise FileNotFoundError("Missing Phase 3.3 input files:\n" + "\n".join(f"- {name}: {PATHS[name]}" for name in missing_inputs) + "\n\nRun notebooks 02, 04, and 04b first with the same RUN_MODE.\n\nAvailable CSV files under results/:\n" + "\n".join(f"  - {p}" for p in available[:80]))

scores_df = pd.read_csv(PATHS["baseline_scores"])
phase31b_metrics_df = pd.read_csv(PATHS["phase31b_candidate_metrics"])
phase31b_selected_df = pd.read_csv(PATHS["phase31b_selected_policies"])
phase31b_thresholds_df = pd.read_csv(PATHS["phase31b_candidate_thresholds"])
five_level_tuned_df = pd.read_csv(PATHS["five_level_tuned"])

missing_columns = sorted(REQUIRED_SCORE_COLUMNS - set(scores_df.columns))
if missing_columns:
    raise ValueError(f"baseline_scores is missing columns: {missing_columns}")

for name, df in [("scores_df", scores_df), ("phase31b_metrics_df", phase31b_metrics_df), ("phase31b_selected_df", phase31b_selected_df), ("phase31b_thresholds_df", phase31b_thresholds_df), ("five_level_tuned_df", five_level_tuned_df)]:
    if "run_mode" in df.columns:
        found_modes = set(df["run_mode"].astype(str).dropna())
        if found_modes and not found_modes.issubset({RUN_MODE}):
            raise ValueError(f"{name} RUN_MODE mismatch. Expected {RUN_MODE}, found {sorted(found_modes)}")

scores_df["split"] = scores_df["split"].astype(str)
scores_df["model"] = scores_df["model"].astype(str)
scores_df["TransactionID"] = scores_df["TransactionID"].astype(str)
scores_df["isFraud"] = scores_df["isFraud"].astype(int)
scores_df["TransactionAmt"] = scores_df["TransactionAmt"].fillna(0).astype(float)
scores_df["score"] = scores_df["score"].astype(float)

available_models = sorted(scores_df["model"].unique())
if SELECTED_RISK_MODEL not in available_models:
    fallback = next((name for name in MODEL_FALLBACK_ORDER if name in available_models), None)
    if fallback is None:
        raise ValueError(f"No supported risk model found. Available models: {available_models}")
    print(f"Selected model {SELECTED_RISK_MODEL!r} not found. Falling back to {fallback!r}.")
    SELECTED_RISK_MODEL = fallback

policy_scores = scores_df[scores_df["model"].eq(SELECTED_RISK_MODEL)].copy()
if not {"validation", "test"}.issubset(set(policy_scores["split"])):
    raise ValueError(f"Selected model {SELECTED_RISK_MODEL!r} must have validation and test rows. Found: {sorted(policy_scores['split'].unique())}")
if policy_scores.duplicated(["split", "TransactionID"]).any():
    dupes = policy_scores[policy_scores.duplicated(["split", "TransactionID"], keep=False)]
    raise ValueError(f"Duplicate score rows detected. Example:\n{dupes.head()}")

validation_scores_df = policy_scores[policy_scores["split"].eq("validation")].reset_index(drop=True)
test_scores_df = policy_scores[policy_scores["split"].eq("test")].reset_index(drop=True)

print(f"Selected risk model: {SELECTED_RISK_MODEL}")
print(f"Validation rows: {len(validation_scores_df):,} | Test rows: {len(test_scores_df):,}")
display(phase31b_selected_df[["selector", "cost_config", "selected_candidate_policy", "selected_candidate_id", "recall_fraud", "precision_fraud", "total_cost", "cost_saving_vs_approve_all"]].sort_values(["selector", "cost_config"]))


Selected risk model: lightgbm_balanced
Validation rows: 15,000 | Test rows: 15,000


,selector,cost_config,selected_candidate_policy,selected_candidate_id,recall_fraud,precision_fraud,total_cost,cost_saving_vs_approve_all
0,level4_tuned_best_cost_selector,Cost-A,level4_bin_strategy_grid,bin_strategy_quantile_60_85_97_cost_a,0.742706,0.278607,19968.487743,25203.599138
1,level4_tuned_best_cost_selector,Cost-B,level4_bin_strategy_grid,bin_strategy_quantile_60_85_97_cost_b,0.742706,0.278607,39936.975486,50407.198276
2,level4_tuned_best_cost_selector,Cost-C,level4_bin_strategy_grid,bin_strategy_quantile_60_85_97_cost_c,0.814324,0.182196,93330.006170,132530.428236
3,level4_tuned_guarded_selector,Cost-A,level4_shrunk_amount_bin_threshold,shrunk_cost_a_lambda_0_75,0.726790,0.302428,20635.042850,24537.044032
4,level4_tuned_guarded_selector,Cost-B,level4_shrunk_amount_bin_threshold,shrunk_cost_b_lambda_0_75,0.726790,0.302428,41270.085699,49074.088063
5,level4_tuned_guarded_selector,Cost-C,level4_bin_strategy_grid,bin_strategy_quantile_60_85_97_cost_c,0.814324,0.182196,93330.006170,132530.428236


## 2. Metrics and Tuned Level 4 Helpers


In [3]:
def safe_average_precision(y_true, scores):
    return float(average_precision_score(y_true, scores)) if len(np.unique(y_true)) >= 2 else np.nan

def safe_roc_auc(y_true, scores):
    return float(roc_auc_score(y_true, scores)) if len(np.unique(y_true)) >= 2 else np.nan

def cost_components(y_true, y_pred, amount, alpha, beta):
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    amount = np.asarray(amount, dtype=float)
    fn_mask = (y_true == 1) & (y_pred == 0)
    fp_mask = (y_true == 0) & (y_pred == 1)
    fn_cost = float(np.sum(amount[fn_mask] * beta))
    fp_cost = float(np.sum(amount[fp_mask] * alpha))
    return fn_cost, fp_cost, fn_cost + fp_cost

def row_cost(y_true, y_pred, amount, alpha, beta):
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    amount = np.asarray(amount, dtype=float)
    return ((y_true == 1) & (y_pred == 0)).astype(float) * amount * beta + ((y_true == 0) & (y_pred == 1)).astype(float) * amount * alpha

def approve_all_cost(y_true, amount, beta):
    y_true = np.asarray(y_true, dtype=int)
    amount = np.asarray(amount, dtype=float)
    return float(np.sum(amount[y_true == 1] * beta))

def evaluate_predictions(policy_name, level, split_name, cost_config_name, y_true, scores, y_pred, amount, alpha, beta, extra=None):
    y_true = np.asarray(y_true, dtype=int)
    scores = np.asarray(scores, dtype=float)
    y_pred = np.asarray(y_pred, dtype=int)
    amount = np.asarray(amount, dtype=float)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    fn_cost, fp_cost, total_cost = cost_components(y_true, y_pred, amount, alpha, beta)
    reference_cost = approve_all_cost(y_true, amount, beta)
    row = {
        "run_mode": RUN_MODE, "sample_rows": SAMPLE_ROWS_LABEL, "level": level, "policy": policy_name,
        "source_model": SELECTED_RISK_MODEL, "split": split_name, "cost_config": cost_config_name,
        "alpha": alpha, "beta": beta, "pr_auc": safe_average_precision(y_true, scores), "roc_auc": safe_roc_auc(y_true, scores),
        "recall_fraud": float(recall_score(y_true, y_pred, zero_division=0)),
        "precision_fraud": float(precision_score(y_true, y_pred, zero_division=0)),
        "f1_fraud": float(f1_score(y_true, y_pred, zero_division=0)),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
        "fn_cost": fn_cost, "fp_cost": fp_cost, "total_cost": total_cost,
        "approve_all_cost": reference_cost,
        "cost_saving_vs_approve_all": reference_cost - total_cost,
        "cost_saving_pct_vs_approve_all": (reference_cost - total_cost) / reference_cost if reference_cost else np.nan,
    }
    if extra:
        row.update(extra)
    return row

def rank_metric_rows(df):
    out = df.copy()
    priority = {"level5_embedding_similarity_threshold_adjustment": 1, "level5_hybrid_score_threshold": 2, "level5_meta_policy_small": 3}
    out["policy_priority"] = out["policy"].map(priority).fillna(99)
    return out.sort_values(["total_cost", "fp_cost", "recall_fraud", "precision_fraud", "policy_priority"], ascending=[True, True, False, False, True])

def safe_id(*parts):
    return "_".join(str(p).lower().replace(".", "_").replace("-", "_").replace(" ", "_") for p in parts)

def assign_bins(amounts, threshold_table):
    table = threshold_table.sort_values("bin_order").reset_index(drop=True)
    edges = list(table["bin_lower_bound"].astype(float)) + [float(table["bin_upper_bound"].iloc[-1])]
    labels = table["amount_bin"].astype(str).tolist()
    bins = pd.cut(
        pd.Series(amounts, dtype="float64"),
        bins=edges,
        labels=labels,
        include_lowest=True,
        right=True,
    ).astype(str).to_numpy()
    if pd.isna(bins).any() or np.any(bins == "nan"):
        raise ValueError("Some rows did not map to an amount bin.")
    return bins

def threshold_table_for_candidate(candidate_id, cost_config_name):
    table = phase31b_thresholds_df[phase31b_thresholds_df["candidate_id"].astype(str).eq(str(candidate_id)) & phase31b_thresholds_df["cost_config"].eq(cost_config_name)].copy()
    if table.empty:
        raise ValueError(f"No threshold table for candidate_id={candidate_id}, cost={cost_config_name}")
    return table.sort_values(["bin_lower_bound", "bin_upper_bound"]).reset_index(drop=True)

def selected_level4_row(cost_config_name, selector=LEVEL4_COMPARATOR_SELECTOR):
    rows = phase31b_selected_df[phase31b_selected_df["selector"].astype(str).eq(selector) & phase31b_selected_df["cost_config"].eq(cost_config_name)].copy()
    if rows.empty and selector != "level4_tuned_best_cost_selector":
        rows = phase31b_selected_df[phase31b_selected_df["selector"].astype(str).eq("level4_tuned_best_cost_selector") & phase31b_selected_df["cost_config"].eq(cost_config_name)].copy()
    if rows.empty:
        raise ValueError(f"No selected Level 4 row found for {cost_config_name}")
    return rows.iloc[0]

def level4_thresholds_for_frame(frame, threshold_table):
    if set(threshold_table["amount_bin"].astype(str)) == {"all"}:
        bins = np.array(["all"] * len(frame), dtype=object)
        thresholds = np.full(len(frame), float(threshold_table["threshold"].iloc[0]), dtype=float)
        return thresholds, bins
    bins = assign_bins(frame["TransactionAmt"].to_numpy(dtype=float), threshold_table)
    lookup = dict(zip(threshold_table["amount_bin"].astype(str), threshold_table["threshold"].astype(float)))
    thresholds = pd.Series(bins).map(lookup).astype(float).to_numpy()
    return thresholds, bins

def apply_level4_thresholds(frame, threshold_table):
    thresholds, bins = level4_thresholds_for_frame(frame, threshold_table)
    pred = (frame["score"].to_numpy(dtype=float) >= thresholds).astype(int)
    return pred, thresholds, bins

for cost_name in COST_CONFIGS:
    row = selected_level4_row(cost_name)
    table = threshold_table_for_candidate(row["selected_candidate_id"], cost_name)
    assert not table.empty
print("Level 4 comparator selector:", LEVEL4_COMPARATOR_SELECTOR)


Level 4 comparator selector: level4_tuned_guarded_selector


## 3. Load Raw IEEE-CIS Rows and Align Splits


In [4]:
TEXT_FEATURE_COLUMNS = [
    "TransactionAmt", "ProductCD",
    "card1", "card2", "card3", "card4", "card5", "card6",
    "addr1", "addr2", "dist1", "dist2",
    "P_emaildomain", "R_emaildomain",
    "C1", "C2", "C5", "C9", "C13", "C14",
    "D1", "D2", "D3", "D4", "D10", "D15",
    "DeviceType", "DeviceInfo", "id_12", "id_15", "id_30", "id_31", "id_33", "id_34",
]
TRANSACTION_TEXT_COLUMNS = [
    "TransactionID", "TransactionDT", "isFraud",
    "TransactionAmt", "ProductCD",
    "card1", "card2", "card3", "card4", "card5", "card6",
    "addr1", "addr2", "dist1", "dist2",
    "P_emaildomain", "R_emaildomain",
    "C1", "C2", "C5", "C9", "C13", "C14",
    "D1", "D2", "D3", "D4", "D10", "D15",
]
IDENTITY_TEXT_COLUMNS = ["TransactionID", "DeviceType", "DeviceInfo", "id_12", "id_15", "id_30", "id_31", "id_33", "id_34"]

missing_raw = [str(PATHS[name]) for name in ["train_transaction", "train_identity"] if not PATHS[name].exists()]
if missing_raw:
    raise FileNotFoundError("Missing raw IEEE-CIS CSV files:\n" + "\n".join(missing_raw))

print("Loading raw transaction rows for neutral text serialization...")
transaction_df = pd.read_csv(PATHS["train_transaction"], usecols=lambda c: c in set(TRANSACTION_TEXT_COLUMNS), nrows=SAMPLE_ROWS)
identity_df = pd.read_csv(PATHS["train_identity"], usecols=lambda c: c in set(IDENTITY_TEXT_COLUMNS))
raw_joined_df = transaction_df.merge(identity_df, on="TransactionID", how="left")
assert len(raw_joined_df) == len(transaction_df), "Left join changed transaction row count."
raw_joined_df = raw_joined_df.sort_values("TransactionDT", kind="mergesort").reset_index(drop=True)
raw_joined_df["TransactionID"] = raw_joined_df["TransactionID"].astype(str)

n_total = len(raw_joined_df)
n_train = int(n_total * 0.70)
n_validation = int(n_total * 0.15)
train_raw_df = raw_joined_df.iloc[:n_train].copy()
validation_raw_df = raw_joined_df.iloc[n_train:n_train + n_validation].copy()
test_raw_df = raw_joined_df.iloc[n_train + n_validation:].copy()

def align_raw_to_scores(raw_split, score_split, split_name):
    raw_indexed = raw_split.set_index("TransactionID", drop=False)
    ids = score_split["TransactionID"].astype(str).tolist()
    missing = [tid for tid in ids if tid not in raw_indexed.index]
    if missing:
        raise ValueError(f"Raw {split_name} split is missing score TransactionIDs. Examples: {missing[:5]}")
    aligned = raw_indexed.loc[ids].reset_index(drop=True)
    if aligned["TransactionID"].astype(str).tolist() != ids:
        raise AssertionError(f"{split_name} alignment failed.")
    if not np.array_equal(aligned["isFraud"].astype(int).to_numpy(), score_split["isFraud"].astype(int).to_numpy()):
        raise AssertionError(f"{split_name} labels differ between raw split and score file.")
    return aligned

validation_raw_df = align_raw_to_scores(validation_raw_df, validation_scores_df, "validation")
test_raw_df = align_raw_to_scores(test_raw_df, test_scores_df, "test")

split_summary = pd.DataFrame([
    {"split": "train", "rows": len(train_raw_df), "fraud_rate": train_raw_df["isFraud"].mean()},
    {"split": "validation", "rows": len(validation_raw_df), "fraud_rate": validation_raw_df["isFraud"].mean()},
    {"split": "test", "rows": len(test_raw_df), "fraud_rate": test_raw_df["isFraud"].mean()},
])
display(split_summary)
for leak_col in ["isFraud", "TransactionID", "TransactionDT"]:
    assert leak_col not in [c for c in TEXT_FEATURE_COLUMNS if c in train_raw_df.columns], f"Text feature leakage: {leak_col}"
del transaction_df, identity_df, raw_joined_df
gc.collect()

Loading raw transaction rows for neutral text serialization...


,split,rows,fraud_rate
0,train,70000,0.026843
1,validation,15000,0.025133
2,test,15000,0.020333


31

## 4. Neutral Table-to-Text and Leakage Audit


In [5]:
FORBIDDEN_TEXT_TOKENS = ["isfraud", "label", "target", "score", "prediction", "predict", "risk", "risky", "suspicious", "fraud", "fraudulent", "high risk", "low risk"]
PRODUCT_INTERP_TOKENS = ["product category", "goods category", "service category", "merchandise"]

def clean_value(value) -> str:
    if pd.isna(value):
        return "missing"
    if isinstance(value, float):
        return str(int(value)) if value.is_integer() else f"{value:.4g}"
    text = str(value).strip().replace("\n", " ")
    return text[:80] if text else "missing"

def serialize_transaction(row: pd.Series) -> str:
    g = lambda c: clean_value(row[c]) if c in row.index else "missing"
    return (
        "Transaction record. "
        f"Amount: {g('TransactionAmt')}. Product code: {g('ProductCD')}. "
        f"Card fields: card1 {g('card1')}, card2 {g('card2')}, card3 {g('card3')}, card4 {g('card4')}, card5 {g('card5')}, card6 {g('card6')}. "
        f"Email domains: purchaser {g('P_emaildomain')}, recipient {g('R_emaildomain')}. "
        f"Address and distance fields: addr1 {g('addr1')}, addr2 {g('addr2')}, dist1 {g('dist1')}, dist2 {g('dist2')}. "
        f"Count fields: C1 {g('C1')}, C2 {g('C2')}, C5 {g('C5')}, C9 {g('C9')}, C13 {g('C13')}, C14 {g('C14')}. "
        f"Time-delta fields: D1 {g('D1')}, D2 {g('D2')}, D3 {g('D3')}, D4 {g('D4')}, D10 {g('D10')}, D15 {g('D15')}. "
        f"Device fields: device type {g('DeviceType')}, device info {g('DeviceInfo')}, identity id_12 {g('id_12')}, id_15 {g('id_15')}, "
        f"operating system {g('id_30')}, browser {g('id_31')}, screen {g('id_33')}, id_34 {g('id_34')}."
    )

def serialize_split_batched(df: pd.DataFrame, split_name: str, chunk_size: int = TEXT_SERIALIZE_CHUNK):
    columns = [c for c in TEXT_FEATURE_COLUMNS if c in df.columns]
    texts = []
    for start in range(0, len(df), chunk_size):
        end = min(start + chunk_size, len(df))
        chunk_texts = df.iloc[start:end][columns].apply(serialize_transaction, axis=1).tolist()
        texts.extend(chunk_texts)
        del chunk_texts
    print(f"Serialized {split_name}: {len(texts):,} texts")
    return texts

def audit_texts(texts, split_name):
    lower = [text.lower() for text in texts]
    forbidden = [tok for tok in FORBIDDEN_TEXT_TOKENS if any(tok in text for text in lower)]
    product_interp = [tok for tok in PRODUCT_INTERP_TOKENS if any(tok in text for text in lower)]
    return {"run_mode": RUN_MODE, "sample_rows": SAMPLE_ROWS_LABEL, "split": split_name, "rows": len(texts), "passed": bool(not forbidden and not product_interp), "forbidden_tokens_found": ";".join(forbidden), "product_interpretation_tokens_found": ";".join(product_interp), "text_policy": "phase33_neutral_structured_table_to_text_no_label_no_score_no_risk_words"}

train_texts = serialize_split_batched(train_raw_df, "train")
validation_texts = serialize_split_batched(validation_raw_df, "validation")
test_texts = serialize_split_batched(test_raw_df, "test")
text_audit_df = pd.DataFrame([audit_texts(train_texts, "train"), audit_texts(validation_texts, "validation"), audit_texts(test_texts, "test")])
if not text_audit_df["passed"].all():
    display(text_audit_df)
    raise AssertionError("Text leakage audit failed.")
TEXT_AUDIT_PATH = RESULTS_DIR / f"phase33_text_leakage_audit_{RUN_OUTPUT_TAG}.csv"
text_audit_df.to_csv(RESULTS_DIR / "phase33_text_leakage_audit.csv", index=False)
text_audit_df.to_csv(TEXT_AUDIT_PATH, index=False)
sample_text_df = pd.DataFrame({"run_mode": RUN_MODE, "split": "train", "TransactionID": train_raw_df["TransactionID"].head(5).astype(str).tolist(), "neutral_text": train_texts[:5]})
sample_text_df.to_csv(RESULTS_DIR / f"phase33_serialized_text_samples_{RUN_OUTPUT_TAG}.csv", index=False)
display(text_audit_df)
display(sample_text_df)

Serialized train: 70,000 texts
Serialized validation: 15,000 texts
Serialized test: 15,000 texts


,run_mode,sample_rows,split,rows,passed,forbidden_tokens_found,product_interpretation_tokens_found,text_policy
0,sample_100k,100000,train,70000,True,,,phase33_neutral_structured_table_to_text_no_la...
1,sample_100k,100000,validation,15000,True,,,phase33_neutral_structured_table_to_text_no_la...
2,sample_100k,100000,test,15000,True,,,phase33_neutral_structured_table_to_text_no_la...


,run_mode,split,TransactionID,neutral_text
0,sample_100k,train,2987000,Transaction record. Amount: 68.5. Product code...
1,sample_100k,train,2987001,Transaction record. Amount: 29. Product code: ...
2,sample_100k,train,2987002,Transaction record. Amount: 59. Product code: ...
3,sample_100k,train,2987003,Transaction record. Amount: 50. Product code: ...
4,sample_100k,train,2987004,Transaction record. Amount: 50. Product code: ...


## 5. Load or Create MiniLM Embeddings


In [6]:
def emb_file(prefix, split):
    return EMBEDDINGS_DIR / f"{prefix}_{split}.npy"

def emb_meta_file(prefix):
    return EMBEDDINGS_DIR / f"{prefix}_metadata.csv"

def emb_ids_file(prefix):
    return EMBEDDINGS_DIR / f"{prefix}_ids.csv"

def expected_split_ids():
    return {"train": train_raw_df["TransactionID"].astype(str).tolist(), "validation": validation_raw_df["TransactionID"].astype(str).tolist(), "test": test_raw_df["TransactionID"].astype(str).tolist()}

def cache_valid(prefix, expected_ids):
    if not all(emb_file(prefix, split).exists() for split in ["train", "validation", "test"]):
        return False
    if not emb_meta_file(prefix).exists() or not emb_ids_file(prefix).exists():
        return False
    try:
        meta = pd.read_csv(emb_meta_file(prefix))
        ids_df = pd.read_csv(emb_ids_file(prefix), dtype={"TransactionID": str})
        for split, ids in expected_ids.items():
            rows = meta[meta["split"].eq(split)]
            if rows.empty or int(rows["rows"].iloc[0]) != len(ids):
                return False
            cached_ids = ids_df[ids_df["split"].eq(split)]["TransactionID"].astype(str).tolist()
            if cached_ids != ids:
                return False
            arr = np.load(emb_file(prefix, split), mmap_mode="r")
            if arr.shape[0] != len(ids):
                return False
        return True
    except Exception as exc:
        print(f"Cache validation failed for {prefix}: {exc}")
        return False

def save_embedding_metadata(prefix, backend, model_name, arrays, expected_ids):
    meta_rows, id_rows = [], []
    for split, arr in arrays.items():
        meta_rows.append({"run_mode": RUN_MODE, "sample_rows": SAMPLE_ROWS_LABEL, "split": split, "rows": int(arr.shape[0]), "embedding_dim": int(arr.shape[1]), "backend": backend, "model_name": model_name, "prefix": prefix})
        id_rows.extend({"split": split, "TransactionID": tid} for tid in expected_ids[split])
    pd.DataFrame(meta_rows).to_csv(emb_meta_file(prefix), index=False)
    pd.DataFrame(id_rows).to_csv(emb_ids_file(prefix), index=False)

def try_import_sentence_transformer():
    try:
        from sentence_transformers import SentenceTransformer
        return SentenceTransformer
    except ImportError:
        if AUTO_INSTALL_SENTENCE_TRANSFORMERS:
            print("Installing sentence-transformers for Colab...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers"])
            from sentence_transformers import SentenceTransformer
            return SentenceTransformer
        return None

def encode_minilm(text_map, prefix):
    SentenceTransformer = try_import_sentence_transformer()
    if SentenceTransformer is None:
        return None
    try:
        model = SentenceTransformer(EMBEDDING_MODEL_NAME)
        arrays = {}
        for split in ["train", "validation", "test"]:
            print(f"Encoding {split} with MiniLM ({len(text_map[split]):,} rows)...")
            arr = model.encode(text_map[split], batch_size=EMBEDDING_BATCH_SIZE, show_progress_bar=True, normalize_embeddings=True, convert_to_numpy=True).astype(np.float32)
            np.save(emb_file(prefix, split), arr)
            arrays[split] = arr
        save_embedding_metadata(prefix, "minilm", EMBEDDING_MODEL_NAME, arrays, expected_ids)
        return arrays
    except Exception as exc:
        print(f"MiniLM embedding failed; falling back to TF-IDF/SVD for smoke verification only: {exc}")
        return None

def encode_tfidf_fallback(text_map, prefix):
    print("Using TF-IDF + TruncatedSVD fallback. This is not valid for final LLM claim.")
    vectorizer = TfidfVectorizer(max_features=TFIDF_MAX_FEATURES, ngram_range=(1, 2), min_df=2)
    x_train = vectorizer.fit_transform(text_map["train"])
    n_components = min(TFIDF_SVD_COMPONENTS, x_train.shape[1] - 1, x_train.shape[0] - 1)
    svd = TruncatedSVD(n_components=n_components, random_state=SEED)
    arrays = {"train": svd.fit_transform(x_train).astype(np.float32), "validation": svd.transform(vectorizer.transform(text_map["validation"])).astype(np.float32), "test": svd.transform(vectorizer.transform(text_map["test"])).astype(np.float32)}
    for split, arr in arrays.items():
        arr = normalize(arr, norm="l2", axis=1).astype(np.float32)
        arrays[split] = arr
        np.save(emb_file(prefix, split), arr)
    save_embedding_metadata(prefix, "tfidf_svd_fallback", "TF-IDF + TruncatedSVD fallback", arrays, expected_ids)
    return arrays

def load_cached_arrays(prefix):
    return {split: np.load(emb_file(prefix, split)).astype(np.float32) for split in ["train", "validation", "test"]}

expected_ids = expected_split_ids()
minilm_prefix = f"phase33_{RUN_OUTPUT_TAG}_minilm"
fallback_prefix = f"phase33_{RUN_OUTPUT_TAG}_tfidf_svd_fallback"
embedding_prefix, embedding_backend = None, None

if cache_valid(minilm_prefix, expected_ids):
    print(f"Using cached MiniLM embeddings: {minilm_prefix}")
    embeddings = load_cached_arrays(minilm_prefix)
    embedding_prefix, embedding_backend = minilm_prefix, "minilm"
else:
    text_map = {"train": train_texts, "validation": validation_texts, "test": test_texts}
    embeddings = encode_minilm(text_map, minilm_prefix)
    if embeddings is not None:
        embedding_prefix, embedding_backend = minilm_prefix, "minilm"
    elif cache_valid(fallback_prefix, expected_ids):
        print(f"Using cached fallback embeddings: {fallback_prefix}")
        embeddings = load_cached_arrays(fallback_prefix)
        embedding_prefix, embedding_backend = fallback_prefix, "tfidf_svd_fallback"
    else:
        embeddings = encode_tfidf_fallback(text_map, fallback_prefix)
        embedding_prefix, embedding_backend = fallback_prefix, "tfidf_svd_fallback"

emb_train = normalize(embeddings["train"], norm="l2", axis=1).astype(np.float32)
emb_validation = normalize(embeddings["validation"], norm="l2", axis=1).astype(np.float32)
emb_test = normalize(embeddings["test"], norm="l2", axis=1).astype(np.float32)
assert emb_train.shape[0] == len(train_raw_df)
assert emb_validation.shape[0] == len(validation_scores_df)
assert emb_test.shape[0] == len(test_scores_df)
del train_texts, validation_texts, test_texts, embeddings
gc.collect()
print(f"Embedding backend used: {embedding_backend}")
print(f"Embedding prefix: {embedding_prefix}")
print(f"Embedding shapes: train={emb_train.shape}, validation={emb_validation.shape}, test={emb_test.shape}")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

Encoding validation with MiniLM (15,000 rows)...


Batches:   0%|          | 0/469 [00:00<?, ?it/s]

Encoding test with MiniLM (15,000 rows)...


Batches:   0%|          | 0/469 [00:00<?, ?it/s]

Embedding backend used: minilm
Embedding prefix: phase33_sample_100k_minilm
Embedding shapes: train=(70000, 384), validation=(15000, 384), test=(15000, 384)


## 6. Train-Only Prototypes and Similarity Features


In [7]:
y_train = train_raw_df["isFraud"].astype(int).to_numpy()
y_validation = validation_scores_df["isFraud"].astype(int).to_numpy()
y_test = test_scores_df["isFraud"].astype(int).to_numpy()
if y_train.sum() == 0 or (len(y_train) - y_train.sum()) == 0:
    raise ValueError("Train split must contain both fraud and legitimate rows to compute prototypes.")

fraud_prototype = normalize(emb_train[y_train == 1].mean(axis=0, keepdims=True), norm="l2", axis=1).astype(np.float32)[0]
legit_prototype = normalize(emb_train[y_train == 0].mean(axis=0, keepdims=True), norm="l2", axis=1).astype(np.float32)[0]

def similarity_delta(emb):
    emb = normalize(emb, norm="l2", axis=1)
    return (emb @ fraud_prototype) - (emb @ legit_prototype)

sim_train = similarity_delta(emb_train)
sim_validation = similarity_delta(emb_validation)
sim_test = similarity_delta(emb_test)
sim_mean = float(np.mean(sim_train))
sim_std = float(np.std(sim_train)) if float(np.std(sim_train)) >= 1e-8 else 1.0
sim_train_z = (sim_train - sim_mean) / sim_std
sim_validation_z = (sim_validation - sim_mean) / sim_std
sim_test_z = (sim_test - sim_mean) / sim_std

similarity_features_df = pd.concat([
    pd.DataFrame({"run_mode": RUN_MODE, "sample_rows": SAMPLE_ROWS_LABEL, "split": "validation", "TransactionID": validation_scores_df["TransactionID"].astype(str), "embedding_backend_used": embedding_backend, "embedding_model_target": EMBEDDING_MODEL_NAME, "similarity_delta": sim_validation, "similarity_delta_z": sim_validation_z, "score": validation_scores_df["score"].to_numpy(dtype=float), "isFraud": y_validation, "TransactionAmt": validation_scores_df["TransactionAmt"].to_numpy(dtype=float)}),
    pd.DataFrame({"run_mode": RUN_MODE, "sample_rows": SAMPLE_ROWS_LABEL, "split": "test", "TransactionID": test_scores_df["TransactionID"].astype(str), "embedding_backend_used": embedding_backend, "embedding_model_target": EMBEDDING_MODEL_NAME, "similarity_delta": sim_test, "similarity_delta_z": sim_test_z, "score": test_scores_df["score"].to_numpy(dtype=float), "isFraud": y_test, "TransactionAmt": test_scores_df["TransactionAmt"].to_numpy(dtype=float)}),
], ignore_index=True)
SIMILARITY_PATH = RESULTS_DIR / f"phase33_similarity_features_{RUN_OUTPUT_TAG}.csv"
similarity_features_df.to_csv(RESULTS_DIR / "phase33_similarity_features.csv", index=False)
similarity_features_df.to_csv(SIMILARITY_PATH, index=False)
display(similarity_features_df.groupby(["split", "isFraud"])[["similarity_delta", "similarity_delta_z", "score"]].agg(["mean", "std", "count"]))


similarity_delta                  similarity_delta_z                      score                 
                               mean       std  count               mean       std  count      mean       std  count
split      isFraud                                                                                                 
test       0              -0.000379  0.004543  14695           0.277186  1.045156  14695  0.203658  0.184606  14695
           1               0.002255  0.005573    305           0.882987  1.282062    305  0.676334  0.306691    305
validation 0              -0.000674  0.004405  14623           0.209263  1.013414  14623  0.185115  0.174258  14623
           1               0.002203  0.005082    377           0.871133  1.169279    377  0.723641  0.280173    377

## 7. Phase 3.3 Advanced Similarity Features

Cell nay mo rong Phase 3.2 similarity delta thanh cac feature giau hon: similarity voi fraud prototype, legit prototype, outlier distance, z-score train-only va validation fit/tune split cho meta-policy.

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

def z_fit(values):
    values = np.asarray(values, dtype=float)
    mean = float(np.mean(values))
    std = float(np.std(values))
    return mean, std if std >= 1e-8 else 1.0

def z_apply(values, params):
    mean, std = params
    return (np.asarray(values, dtype=float) - mean) / std

def proto_similarity(emb, proto):
    emb = normalize(emb, norm="l2", axis=1)
    return emb @ proto

sim_fraud_train = proto_similarity(emb_train, fraud_prototype)
sim_legit_train = proto_similarity(emb_train, legit_prototype)
outlier_train = 1.0 - sim_legit_train
amount_log_train = np.log1p(train_raw_df["TransactionAmt"].astype(float).to_numpy())

sim_fraud_validation = proto_similarity(emb_validation, fraud_prototype)
sim_legit_validation = proto_similarity(emb_validation, legit_prototype)
outlier_validation = 1.0 - sim_legit_validation
amount_log_validation = np.log1p(validation_scores_df["TransactionAmt"].astype(float).to_numpy())

sim_fraud_test = proto_similarity(emb_test, fraud_prototype)
sim_legit_test = proto_similarity(emb_test, legit_prototype)
outlier_test = 1.0 - sim_legit_test
amount_log_test = np.log1p(test_scores_df["TransactionAmt"].astype(float).to_numpy())

z_params = {
    "sim_fraud": z_fit(sim_fraud_train),
    "sim_legit": z_fit(sim_legit_train),
    "similarity_delta": z_fit(sim_train),
    "outlier": z_fit(outlier_train),
    "amount_log": z_fit(amount_log_train),
}

validation_df = validation_scores_df.copy()
test_df = test_scores_df.copy()
validation_df["sim_fraud"] = sim_fraud_validation
validation_df["sim_legit"] = sim_legit_validation
validation_df["similarity_delta"] = sim_validation
validation_df["outlier_score"] = outlier_validation
validation_df["sim_fraud_z"] = z_apply(sim_fraud_validation, z_params["sim_fraud"])
validation_df["sim_legit_z"] = z_apply(sim_legit_validation, z_params["sim_legit"])
validation_df["similarity_delta_z"] = z_apply(sim_validation, z_params["similarity_delta"])
validation_df["outlier_score_z"] = z_apply(outlier_validation, z_params["outlier"])
validation_df["amount_log_z"] = z_apply(amount_log_validation, z_params["amount_log"])

test_df["sim_fraud"] = sim_fraud_test
test_df["sim_legit"] = sim_legit_test
test_df["similarity_delta"] = sim_test
test_df["outlier_score"] = outlier_test
test_df["sim_fraud_z"] = z_apply(sim_fraud_test, z_params["sim_fraud"])
test_df["sim_legit_z"] = z_apply(sim_legit_test, z_params["sim_legit"])
test_df["similarity_delta_z"] = z_apply(sim_test, z_params["similarity_delta"])
test_df["outlier_score_z"] = z_apply(outlier_test, z_params["outlier"])
test_df["amount_log_z"] = z_apply(amount_log_test, z_params["amount_log"])

# Keep Phase 2 validation order, which follows the time-based split order.
validation_df = validation_df.reset_index(drop=True)
validation_fit_end = len(validation_df) // 2
validation_df["validation_role"] = np.where(np.arange(len(validation_df)) < validation_fit_end, "validation_fit", "validation_tune")
validation_fit_df = validation_df[validation_df["validation_role"].eq("validation_fit")].copy().reset_index(drop=True)
validation_tune_df = validation_df[validation_df["validation_role"].eq("validation_tune")].copy().reset_index(drop=True)

similarity_features_v2_df = pd.concat([
    pd.DataFrame({"run_mode": RUN_MODE, "sample_rows": SAMPLE_ROWS_LABEL, "split": "validation", "TransactionID": validation_df["TransactionID"], "embedding_backend_used": embedding_backend, "sim_fraud": validation_df["sim_fraud"], "sim_legit": validation_df["sim_legit"], "similarity_delta": validation_df["similarity_delta"], "outlier_score": validation_df["outlier_score"], "sim_fraud_z": validation_df["sim_fraud_z"], "sim_legit_z": validation_df["sim_legit_z"], "similarity_delta_z": validation_df["similarity_delta_z"], "outlier_score_z": validation_df["outlier_score_z"], "amount_log_z": validation_df["amount_log_z"]}),
    pd.DataFrame({"run_mode": RUN_MODE, "sample_rows": SAMPLE_ROWS_LABEL, "split": "test", "TransactionID": test_df["TransactionID"], "embedding_backend_used": embedding_backend, "sim_fraud": test_df["sim_fraud"], "sim_legit": test_df["sim_legit"], "similarity_delta": test_df["similarity_delta"], "outlier_score": test_df["outlier_score"], "sim_fraud_z": test_df["sim_fraud_z"], "sim_legit_z": test_df["sim_legit_z"], "similarity_delta_z": test_df["similarity_delta_z"], "outlier_score_z": test_df["outlier_score_z"], "amount_log_z": test_df["amount_log_z"]}),
], ignore_index=True)
SIMILARITY_V2_PATH = RESULTS_DIR / f"phase33_similarity_features_{RUN_OUTPUT_TAG}.csv"
similarity_features_v2_df.to_csv(RESULTS_DIR / "phase33_similarity_features.csv", index=False)
similarity_features_v2_df.to_csv(SIMILARITY_V2_PATH, index=False)

display(validation_df.groupby("validation_role")["isFraud"].agg(["count", "sum", "mean"]))
display(similarity_features_v2_df.groupby(["split"])[["sim_fraud", "similarity_delta", "outlier_score"]].agg(["mean", "std", "min", "max"]))

,count,sum,mean
validation_role,,,
validation_fit,7500,197,0.026267
validation_tune,7500,180,0.024000


sim_fraud                               similarity_delta                               outlier_score                              
                mean       std       min       max             mean       std       min       max          mean       std       min       max
split                                                                                                                                        
test        0.961711  0.023872  0.815579  0.989615        -0.000325  0.004581 -0.009562  0.015931      0.037964  0.025965  0.008578  0.195414
validation  0.962837  0.022974  0.821081  0.990589        -0.000601  0.004446 -0.010197  0.014931      0.036562  0.024954  0.008402  0.189216

## 8. Candidate Families A-D

Tat ca candidate duoc tune tren `validation_tune`; test chi duoc dung sau khi candidate da duoc chon.

In [ ]:
PHASE33_GAMMA_GRID = [-0.30, -0.20, -0.10, -0.05, 0.00, 0.05, 0.10, 0.20, 0.30]
W_LGB_GRID = [0.75, 1.00, 1.25]
W_SIM_GRID = [-1.00, -0.50, 0.00, 0.50, 1.00]
W_AMT_GRID = [-0.50, 0.00, 0.50]
OUTLIER_SIM_CUTOFF_GRID = [-0.5, 0.0, 0.5, 1.0]
OUTLIER_CUTOFF_GRID = [-0.5, 0.0, 0.5, 1.0]
OUTLIER_DELTA_GRID = [0.03, 0.05, 0.10, 0.15]

phase33_candidate_rows = []
phase33_prediction_store = {}

def register_phase33_candidate(cost_config_name, policy_name, candidate_id, family, val_scores, val_pred, test_scores, test_pred, extra=None):
    alpha, beta = COST_CONFIGS[cost_config_name]["alpha"], COST_CONFIGS[cost_config_name]["beta"]
    base_extra = {
        "candidate_id": candidate_id,
        "candidate_family": family,
        "selection_pool": True,
        "uses_embedding": True,
        "embedding_backend": embedding_backend,
        "embedding_model_target": EMBEDDING_MODEL_NAME,
        "selection_status": "candidate_validation_tune",
    }
    if extra:
        base_extra.update(extra)
    val_row = evaluate_predictions(policy_name, 5, "validation_tune", cost_config_name, validation_tune_df["isFraud"].to_numpy(dtype=int), val_scores, val_pred, validation_tune_df["TransactionAmt"].to_numpy(dtype=float), alpha, beta, extra=base_extra)
    test_extra = dict(base_extra)
    test_extra["selection_pool"] = False
    test_extra["selection_status"] = "candidate_test_after_validation_tuning"
    test_row = evaluate_predictions(policy_name, 5, "test", cost_config_name, test_df["isFraud"].to_numpy(dtype=int), test_scores, test_pred, test_df["TransactionAmt"].to_numpy(dtype=float), alpha, beta, extra=test_extra)
    phase33_candidate_rows.extend([val_row, test_row])
    phase33_prediction_store[(cost_config_name, candidate_id, "validation_tune")] = {"score": np.asarray(val_scores), "pred": np.asarray(val_pred)}
    phase33_prediction_store[(cost_config_name, candidate_id, "test")] = {"score": np.asarray(test_scores), "pred": np.asarray(test_pred)}

def tune_threshold(frame, scores, cost_config_name):
    alpha, beta = COST_CONFIGS[cost_config_name]["alpha"], COST_CONFIGS[cost_config_name]["beta"]
    y = frame["isFraud"].to_numpy(dtype=int)
    amount = frame["TransactionAmt"].to_numpy(dtype=float)
    rows = []
    for threshold in THRESHOLD_GRID:
        pred = (scores >= threshold).astype(int)
        fn_cost, fp_cost, total_cost = cost_components(y, pred, amount, alpha, beta)
        rows.append({"threshold": float(threshold), "total_cost": total_cost, "fp_cost": fp_cost, "recall_fraud": float(recall_score(y, pred, zero_division=0)), "precision_fraud": float(precision_score(y, pred, zero_division=0))})
    return pd.DataFrame(rows).sort_values(["total_cost", "fp_cost", "recall_fraud", "precision_fraud", "threshold"], ascending=[True, True, False, False, True]).iloc[0]

def phase33_base_thresholds(cost_config_name):
    selected_l4 = selected_level4_row(cost_config_name)
    table = threshold_table_for_candidate(selected_l4["selected_candidate_id"], cost_config_name)
    val_base, val_bins = level4_thresholds_for_frame(validation_tune_df, table)
    test_base, test_bins = level4_thresholds_for_frame(test_df, table)
    return selected_l4, val_base, val_bins, test_base, test_bins

def phase33_safe_id(*parts):
    return safe_id(*parts)

# Candidate A: train-only prototype threshold adjustment.
for cost_config_name in COST_CONFIGS:
    selected_l4, val_base, val_bins, test_base, test_bins = phase33_base_thresholds(cost_config_name)
    for feature_name, policy_name in [
        ("similarity_delta_z", "level5_v2_train_fraud_minus_legit_adjustment"),
        ("sim_fraud_z", "level5_v2_train_fraud_prototype_adjustment"),
    ]:
        for gamma in PHASE33_GAMMA_GRID:
            val_threshold = np.clip(val_base - gamma * validation_tune_df[feature_name].to_numpy(dtype=float), 0.0, 1.0)
            test_threshold = np.clip(test_base - gamma * test_df[feature_name].to_numpy(dtype=float), 0.0, 1.0)
            val_pred = (validation_tune_df["score"].to_numpy(dtype=float) >= val_threshold).astype(int)
            test_pred = (test_df["score"].to_numpy(dtype=float) >= test_threshold).astype(int)
            candidate_id = phase33_safe_id("proto", cost_config_name, feature_name, f"gamma_{gamma:.2f}")
            register_phase33_candidate(cost_config_name, policy_name, candidate_id, "prototype_threshold_adjustment", validation_tune_df["score"].to_numpy(dtype=float) - val_threshold, val_pred, test_df["score"].to_numpy(dtype=float) - test_threshold, test_pred, extra={"gamma": float(gamma), "feature_name": feature_name, "level4_selected_candidate_id": selected_l4["selected_candidate_id"], "threshold_min": float(np.min(test_threshold)), "threshold_max": float(np.max(test_threshold))})

# Candidate B: hybrid score fusion.
def phase33_hybrid_score(frame, w_lgb, w_sim, w_amt):
    base = np.clip(frame["score"].to_numpy(dtype=float), 1e-6, 1 - 1e-6)
    linear = w_lgb * logit(base) + w_sim * frame["similarity_delta_z"].to_numpy(dtype=float) + w_amt * frame["amount_log_z"].to_numpy(dtype=float)
    return expit(linear)

for cost_config_name in COST_CONFIGS:
    for w_lgb in W_LGB_GRID:
        for w_sim in W_SIM_GRID:
            for w_amt in W_AMT_GRID:
                val_score = phase33_hybrid_score(validation_tune_df, w_lgb, w_sim, w_amt)
                test_score = phase33_hybrid_score(test_df, w_lgb, w_sim, w_amt)
                threshold = float(tune_threshold(validation_tune_df, val_score, cost_config_name)["threshold"])
                val_pred = (val_score >= threshold).astype(int)
                test_pred = (test_score >= threshold).astype(int)
                candidate_id = phase33_safe_id("fusion", cost_config_name, f"lgb_{w_lgb:.2f}", f"sim_{w_sim:.2f}", f"amt_{w_amt:.2f}", f"thr_{threshold:.3f}")
                register_phase33_candidate(cost_config_name, "level5_v2_hybrid_score_fusion", candidate_id, "hybrid_score_fusion", val_score, val_pred, test_score, test_pred, extra={"w_lgb": float(w_lgb), "w_sim": float(w_sim), "w_amt": float(w_amt), "threshold": threshold, "amount_bin_count": 1})

# Candidate C: compact logistic meta-policy, fit on validation_fit only because train scores are unavailable.
META_FEATURES = ["score", "sim_fraud_z", "sim_legit_z", "similarity_delta_z", "outlier_score_z", "amount_log_z"]
def meta_matrix(frame):
    return frame[META_FEATURES].to_numpy(dtype=float)

meta_scaler = StandardScaler()
X_fit = meta_scaler.fit_transform(meta_matrix(validation_fit_df))
X_tune = meta_scaler.transform(meta_matrix(validation_tune_df))
X_test = meta_scaler.transform(meta_matrix(test_df))
y_fit = validation_fit_df["isFraud"].to_numpy(dtype=int)
meta_model = LogisticRegression(class_weight="balanced", solver="liblinear", max_iter=1000, random_state=SEED)
meta_model.fit(X_fit, y_fit)
meta_val_score = meta_model.predict_proba(X_tune)[:, 1]
meta_test_score = meta_model.predict_proba(X_test)[:, 1]
phase33_feature_importance_df = pd.DataFrame({"feature": META_FEATURES, "coefficient": meta_model.coef_[0], "abs_coefficient": np.abs(meta_model.coef_[0]), "policy": "level5_v2_logistic_meta_policy"}).sort_values("abs_coefficient", ascending=False)

for cost_config_name in COST_CONFIGS:
    threshold = float(tune_threshold(validation_tune_df, meta_val_score, cost_config_name)["threshold"])
    val_pred = (meta_val_score >= threshold).astype(int)
    test_pred = (meta_test_score >= threshold).astype(int)
    candidate_id = phase33_safe_id("meta_logreg", cost_config_name, f"thr_{threshold:.3f}")
    register_phase33_candidate(cost_config_name, "level5_v2_logistic_meta_policy", candidate_id, "compact_logistic_meta_policy", meta_val_score, val_pred, meta_test_score, test_pred, extra={"threshold": threshold, "fit_source": "validation_fit_no_train_scores_available", "feature_count": len(META_FEATURES)})

# Candidate D: contextual outlier adjustment.
for cost_config_name in COST_CONFIGS:
    selected_l4, val_base, val_bins, test_base, test_bins = phase33_base_thresholds(cost_config_name)
    for sim_cutoff in OUTLIER_SIM_CUTOFF_GRID:
        for outlier_cutoff in OUTLIER_CUTOFF_GRID:
            for delta in OUTLIER_DELTA_GRID:
                val_condition = (validation_tune_df["similarity_delta_z"].to_numpy(dtype=float) > sim_cutoff) & (validation_tune_df["outlier_score_z"].to_numpy(dtype=float) > outlier_cutoff)
                test_condition = (test_df["similarity_delta_z"].to_numpy(dtype=float) > sim_cutoff) & (test_df["outlier_score_z"].to_numpy(dtype=float) > outlier_cutoff)
                val_threshold = np.clip(val_base - np.where(val_condition, delta, 0.0), 0.0, 1.0)
                test_threshold = np.clip(test_base - np.where(test_condition, delta, 0.0), 0.0, 1.0)
                val_pred = (validation_tune_df["score"].to_numpy(dtype=float) >= val_threshold).astype(int)
                test_pred = (test_df["score"].to_numpy(dtype=float) >= test_threshold).astype(int)
                candidate_id = phase33_safe_id("outlier", cost_config_name, f"sim_{sim_cutoff:.1f}", f"out_{outlier_cutoff:.1f}", f"delta_{delta:.2f}")
                register_phase33_candidate(cost_config_name, "level5_v2_contextual_outlier_adjustment", candidate_id, "contextual_outlier_threshold_adjustment", validation_tune_df["score"].to_numpy(dtype=float) - val_threshold, val_pred, test_df["score"].to_numpy(dtype=float) - test_threshold, test_pred, extra={"sim_cutoff": float(sim_cutoff), "outlier_cutoff": float(outlier_cutoff), "delta": float(delta), "threshold_min": float(np.min(test_threshold)), "threshold_max": float(np.max(test_threshold))})

phase33_candidate_metrics_df = pd.DataFrame(phase33_candidate_rows)
display(phase33_candidate_metrics_df[phase33_candidate_metrics_df["split"].eq("validation_tune")].sort_values(["cost_config", "total_cost"]).groupby("cost_config").head(8)[["cost_config", "policy", "candidate_family", "candidate_id", "recall_fraud", "precision_fraud", "fp_cost", "total_cost"]])
display(phase33_feature_importance_df)
print(f"Phase 3.3 candidate metric rows: {len(phase33_candidate_metrics_df):,}")

## 9. Select Level 5 v2 on Validation and Evaluate Frozen Test Policy

In [ ]:
PHASE33_SIMPLICITY_RANK = {
    "prototype_threshold_adjustment": 1,
    "hybrid_score_fusion": 2,
    "compact_logistic_meta_policy": 3,
    "contextual_outlier_threshold_adjustment": 4,
}

selected_validation_rows = []
selected_test_rows = []
phase33_selected_test_actions = {}
phase33_level4_test_actions = {}

phase33_candidate_metrics_df["simplicity_rank"] = phase33_candidate_metrics_df["candidate_family"].map(PHASE33_SIMPLICITY_RANK).fillna(99)
for cost_config_name in COST_CONFIGS:
    pool = phase33_candidate_metrics_df[phase33_candidate_metrics_df["split"].eq("validation_tune") & phase33_candidate_metrics_df["cost_config"].eq(cost_config_name)].copy()
    pool = pool.sort_values(["total_cost", "fp_cost", "recall_fraud", "precision_fraud", "simplicity_rank"], ascending=[True, True, False, False, True])
    selected_val = pool.iloc[0].copy()
    selected_candidate_id = str(selected_val["candidate_id"])
    selected_test = phase33_candidate_metrics_df[phase33_candidate_metrics_df["split"].eq("test") & phase33_candidate_metrics_df["cost_config"].eq(cost_config_name) & phase33_candidate_metrics_df["candidate_id"].astype(str).eq(selected_candidate_id)].iloc[0].copy()
    selected_val["selector"] = "phase33_validation_tune_total_cost_selector"
    selected_val["selected_candidate_id"] = selected_candidate_id
    selected_val["selected_candidate_policy"] = selected_val["policy"]
    selected_val["selected_validation_total_cost"] = float(selected_val["total_cost"])
    selected_val["selection_status"] = "selected_validation_tune"
    selected_test["selector"] = "phase33_validation_tune_total_cost_selector"
    selected_test["selected_candidate_id"] = selected_candidate_id
    selected_test["selected_candidate_policy"] = selected_val["policy"]
    selected_test["selected_validation_total_cost"] = float(selected_val["total_cost"])
    selected_test["selection_status"] = "selected_frozen_test"
    selected_validation_rows.append(selected_val.to_dict())
    selected_test_rows.append(selected_test.to_dict())
    phase33_selected_test_actions[cost_config_name] = phase33_prediction_store[(cost_config_name, selected_candidate_id, "test")]["pred"]
    l4_row = selected_level4_row(cost_config_name)
    l4_table = threshold_table_for_candidate(l4_row["selected_candidate_id"], cost_config_name)
    l4_pred, _, _ = apply_level4_thresholds(test_df, l4_table)
    phase33_level4_test_actions[cost_config_name] = l4_pred

phase33_selected_policy_df = pd.concat([pd.DataFrame(selected_validation_rows), pd.DataFrame(selected_test_rows)], ignore_index=True, sort=False)
display(phase33_selected_policy_df[["split", "cost_config", "policy", "candidate_family", "candidate_id", "recall_fraud", "precision_fraud", "fp_cost", "total_cost", "cost_saving_vs_approve_all", "selection_status"]])

## 10. Compare Level 5 v2 Against Tuned Level 4 and Level 5 v1

In [ ]:
level4_guarded_test = phase31b_metrics_df[phase31b_metrics_df["split"].astype(str).eq("test") & phase31b_metrics_df["policy"].astype(str).eq(LEVEL4_COMPARATOR_SELECTOR)].copy()
if level4_guarded_test.empty:
    level4_guarded_test = five_level_tuned_df[five_level_tuned_df["split"].astype(str).eq("test") & five_level_tuned_df["policy"].astype(str).eq(LEVEL4_COMPARATOR_SELECTOR)].copy()
if level4_guarded_test.empty:
    raise ValueError("Cannot find tuned Level 4 comparator rows.")

phase33_selected_test_df = phase33_selected_policy_df[phase33_selected_policy_df["split"].astype(str).eq("test")].copy()
summary_compare_v2 = phase33_selected_test_df.merge(level4_guarded_test[["cost_config", "total_cost", "cost_saving_vs_approve_all", "recall_fraud", "precision_fraud"]], on="cost_config", how="left", suffixes=("_level5_v2", "_level4"))
summary_compare_v2["total_cost_delta_level5_v2_minus_level4"] = summary_compare_v2["total_cost_level5_v2"] - summary_compare_v2["total_cost_level4"]
summary_compare_v2["saving_delta_level5_v2_minus_level4"] = summary_compare_v2["cost_saving_vs_approve_all_level5_v2"] - summary_compare_v2["cost_saving_vs_approve_all_level4"]
summary_compare_v2["level5_v2_beats_level4"] = summary_compare_v2["total_cost_delta_level5_v2_minus_level4"] < 0
phase33_win_count = int(summary_compare_v2["level5_v2_beats_level4"].sum())
phase33_claim_allowed = bool(embedding_backend == "minilm" and phase33_win_count >= 2)

phase33_level5_for_comparison = phase33_selected_test_df.copy()
phase33_level5_for_comparison["model"] = "level5_v2_advanced_hybrid"
phase33_level5_for_comparison["status"] = "complete_phase33_source_generated"
phase33_level5_for_comparison["level_label"] = "Level 5 v2 - Advanced LLM-augmented hybrid policy"
phase33_level5_for_comparison["algorithm"] = "lightgbm_score_plus_minilm_similarity_meta_policy"
phase33_level5_for_comparison["exploration"] = "none"
phase33_level5_for_comparison["uses_embedding"] = True
phase33_level5_for_comparison["embedding_backend"] = embedding_backend

level5_v1_path = RESULTS_DIR / f"five_level_comparison_level5_{RUN_OUTPUT_TAG}.csv"
if level5_v1_path.exists():
    base_comparison_v2_df = pd.read_csv(level5_v1_path)
else:
    base_comparison_v2_df = five_level_tuned_df.copy()
comparison_level5_v2_df = pd.concat([base_comparison_v2_df, phase33_level5_for_comparison], ignore_index=True, sort=False)

print(f"Level 5 v2 beats tuned Level 4 in {phase33_win_count}/3 cost settings.")
print(f"Embedding backend used: {embedding_backend}; final LLM claim allowed: {phase33_claim_allowed}")
display(summary_compare_v2[["cost_config", "policy", "candidate_family", "total_cost_level4", "total_cost_level5_v2", "total_cost_delta_level5_v2_minus_level4", "saving_delta_level5_v2_minus_level4", "level5_v2_beats_level4", "recall_fraud_level4", "recall_fraud_level5_v2", "precision_fraud_level4", "precision_fraud_level5_v2"]])

## 11. Disagreement Analysis

In [ ]:
def change_label(y, l4_action, l5_action):
    if l4_action == l5_action:
        return "same"
    if y == 1 and l4_action == 0 and l5_action == 1:
        return "improved_caught_fraud"
    if y == 1 and l4_action == 1 and l5_action == 0:
        return "worse_missed_fraud"
    if y == 0 and l4_action == 0 and l5_action == 1:
        return "new_false_positive"
    if y == 0 and l4_action == 1 and l5_action == 0:
        return "avoided_false_positive"
    return "other"

disagreement_rows = []
for cost_config_name, cfg in COST_CONFIGS.items():
    alpha, beta = cfg["alpha"], cfg["beta"]
    l4_pred = phase33_level4_test_actions[cost_config_name]
    l5_pred = phase33_selected_test_actions[cost_config_name]
    y = test_df["isFraud"].to_numpy(dtype=int)
    amount = test_df["TransactionAmt"].to_numpy(dtype=float)
    l4_cost = row_cost(y, l4_pred, amount, alpha, beta)
    l5_cost = row_cost(y, l5_pred, amount, alpha, beta)
    mask = l4_pred != l5_pred
    if not np.any(mask):
        continue
    selected_l5 = phase33_selected_test_df[phase33_selected_test_df["cost_config"].eq(cost_config_name)].iloc[0]
    diff = test_df.loc[mask, ["TransactionID", "isFraud", "TransactionAmt", "score", "sim_fraud_z", "similarity_delta_z", "outlier_score_z"]].copy()
    diff["cost_config"] = cost_config_name
    diff["alpha"] = alpha
    diff["beta"] = beta
    diff["level4_action"] = l4_pred[mask]
    diff["level5_v2_action"] = l5_pred[mask]
    diff["level4_row_cost"] = l4_cost[mask]
    diff["level5_v2_row_cost"] = l5_cost[mask]
    diff["cost_delta_level5_v2_minus_level4"] = diff["level5_v2_row_cost"] - diff["level4_row_cost"]
    diff["change_type"] = [change_label(int(yy), int(a4), int(a5)) for yy, a4, a5 in zip(y[mask], l4_pred[mask], l5_pred[mask])]
    diff["level5_v2_policy"] = selected_l5["policy"]
    diff["level5_v2_candidate_id"] = selected_l5["candidate_id"]
    disagreement_rows.append(diff)

if disagreement_rows:
    phase33_disagreement_df = pd.concat(disagreement_rows, ignore_index=True).sort_values(["cost_config", "cost_delta_level5_v2_minus_level4", "TransactionAmt"], ascending=[True, True, False])
else:
    phase33_disagreement_df = pd.DataFrame(columns=["TransactionID", "isFraud", "TransactionAmt", "score", "similarity_delta_z", "cost_config", "level4_action", "level5_v2_action", "level4_row_cost", "level5_v2_row_cost", "cost_delta_level5_v2_minus_level4", "change_type"])

phase33_disagreement_summary_df = phase33_disagreement_df.groupby(["cost_config", "change_type"], dropna=False).agg(rows=("TransactionID", "count"), total_cost_delta=("cost_delta_level5_v2_minus_level4", "sum"), mean_amount=("TransactionAmt", "mean")).reset_index() if not phase33_disagreement_df.empty else pd.DataFrame()
display(phase33_disagreement_summary_df)
display(phase33_disagreement_df.head(30))

## 12. Save Phase 3.3 Outputs and Figures

In [ ]:
PHASE33_CANDIDATE_METRICS_PATH = RESULTS_DIR / f"phase33_llm_hybrid_metrics_{RUN_OUTPUT_TAG}.csv"
PHASE33_SELECTED_POLICY_PATH = RESULTS_DIR / f"phase33_selected_policy_{RUN_OUTPUT_TAG}.csv"
PHASE33_DISAGREEMENT_PATH = RESULTS_DIR / f"phase33_disagreement_analysis_{RUN_OUTPUT_TAG}.csv"
PHASE33_COMPARISON_PATH = RESULTS_DIR / f"five_level_comparison_level5_v2_{RUN_OUTPUT_TAG}.csv"
PHASE33_WIN_SUMMARY_PATH = RESULTS_DIR / f"phase33_level5_v2_vs_level4_{RUN_OUTPUT_TAG}.csv"
PHASE33_FEATURE_IMPORTANCE_PATH = RESULTS_DIR / f"phase33_feature_importance_{RUN_OUTPUT_TAG}.csv"
PHASE33_METADATA_PATH = RESULTS_DIR / f"phase33_run_metadata_{RUN_OUTPUT_TAG}.json"

phase33_candidate_metrics_df.to_csv(RESULTS_DIR / "phase33_llm_hybrid_metrics.csv", index=False)
phase33_candidate_metrics_df.to_csv(PHASE33_CANDIDATE_METRICS_PATH, index=False)
phase33_selected_policy_df.to_csv(RESULTS_DIR / "phase33_selected_policy.csv", index=False)
phase33_selected_policy_df.to_csv(PHASE33_SELECTED_POLICY_PATH, index=False)
phase33_disagreement_df.to_csv(RESULTS_DIR / "phase33_disagreement_analysis.csv", index=False)
phase33_disagreement_df.to_csv(PHASE33_DISAGREEMENT_PATH, index=False)
comparison_level5_v2_df.to_csv(RESULTS_DIR / "five_level_comparison_level5_v2.csv", index=False)
comparison_level5_v2_df.to_csv(PHASE33_COMPARISON_PATH, index=False)
summary_compare_v2.to_csv(RESULTS_DIR / "phase33_level5_v2_vs_level4.csv", index=False)
summary_compare_v2.to_csv(PHASE33_WIN_SUMMARY_PATH, index=False)
phase33_feature_importance_df.to_csv(RESULTS_DIR / "phase33_feature_importance.csv", index=False)
phase33_feature_importance_df.to_csv(PHASE33_FEATURE_IMPORTANCE_PATH, index=False)

plot_df = pd.concat([
    level4_guarded_test.assign(policy_short="Level4 tuned dynamic"),
    phase33_selected_test_df.assign(policy_short="Level5 v2 hybrid"),
], ignore_index=True, sort=False)

plt.figure(figsize=(10, 5))
sns.barplot(data=plot_df, x="cost_config", y="total_cost", hue="policy_short")
plt.title("Phase 3.3 Total Cost - Level 4 vs Level 5 v2")
plt.ylabel("Total Cost")
plt.xlabel("Cost configuration")
plt.tight_layout()
plt.savefig(FIGURES_DIR / f"phase33_level4_vs_level5_v2_total_cost_{RUN_OUTPUT_TAG}.png", dpi=160)
plt.show()

plt.figure(figsize=(10, 5))
sns.barplot(data=plot_df, x="cost_config", y="cost_saving_vs_approve_all", hue="policy_short")
plt.title("Phase 3.3 Cost Saving - Level 4 vs Level 5 v2")
plt.ylabel("Cost Saving vs approve-all")
plt.xlabel("Cost configuration")
plt.tight_layout()
plt.savefig(FIGURES_DIR / f"phase33_cost_saving_{RUN_OUTPUT_TAG}.png", dpi=160)
plt.show()

if not phase33_disagreement_summary_df.empty:
    plt.figure(figsize=(11, 5))
    sns.barplot(data=phase33_disagreement_summary_df, x="cost_config", y="total_cost_delta", hue="change_type")
    plt.axhline(0, color="black", linewidth=1)
    plt.title("Phase 3.3 Disagreement Impact (Level 5 v2 - Level 4)")
    plt.ylabel("Total Cost Delta")
    plt.xlabel("Cost configuration")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / f"phase33_disagreement_impact_{RUN_OUTPUT_TAG}.png", dpi=160)
    plt.show()

phase33_metadata = {
    "phase": "3.3",
    "run_mode": RUN_MODE,
    "sample_rows": SAMPLE_ROWS_LABEL,
    "project_root": str(PROJECT_ROOT),
    "selected_risk_model": SELECTED_RISK_MODEL,
    "level4_comparator_selector": LEVEL4_COMPARATOR_SELECTOR,
    "embedding_model_target": EMBEDDING_MODEL_NAME,
    "embedding_backend_used": embedding_backend,
    "embedding_prefix": embedding_prefix,
    "candidate_families": sorted(phase33_candidate_metrics_df["candidate_family"].dropna().unique().tolist()),
    "candidate_metric_rows": int(len(phase33_candidate_metrics_df)),
    "selected_on": "validation_tune",
    "test_usage": "report_only_after_selection",
    "level5_v2_beats_level4_count": int(phase33_win_count),
    "level5_v2_beats_level4_out_of": 3,
    "final_llm_claim_allowed": phase33_claim_allowed,
    "claim_guard": "LLM claim allowed only when backend is minilm and Level 5 v2 beats tuned Level 4 in at least 2/3 cost settings.",
    "created_at": pd.Timestamp.now().isoformat(),
}
PHASE33_METADATA_PATH.write_text(json.dumps(phase33_metadata, indent=2), encoding="utf-8")

print("Saved Phase 3.3 outputs:")
for path in [PHASE33_CANDIDATE_METRICS_PATH, PHASE33_SELECTED_POLICY_PATH, SIMILARITY_V2_PATH, PHASE33_DISAGREEMENT_PATH, PHASE33_FEATURE_IMPORTANCE_PATH, PHASE33_COMPARISON_PATH, PHASE33_WIN_SUMMARY_PATH, PHASE33_METADATA_PATH]:
    print(" -", path)

## 13. Claim Guard for Report

In [ ]:
if embedding_backend == "minilm":
    print("MiniLM embeddings were used; this run is eligible for LLM-assisted claims.")
else:
    print("Fallback embeddings were used; this run verifies code flow only and must not be used as final LLM evidence.")

if phase33_claim_allowed:
    print(f"Level 5 v2 beats tuned Level 4 in {phase33_win_count}/3 cost settings. Claim narrowly on this run mode, split, cost matrix, and metric set.")
else:
    print(f"Level 5 v2 beats tuned Level 4 in {phase33_win_count}/3 cost settings. Do not claim broad LLM superiority; keep Level 4 as main contribution if needed.")

display(summary_compare_v2)